In [97]:
#Imports
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import re
import joblib
import numpy as np

In [98]:
#Load Dataset
data = pd.read_csv("cefrj-vocabulary.csv")
dd = pd.read_csv("Vocabulary_frequency.csv")
#dd1 = pd.read_csv("words.csv")
#dd2 = pd.read_csv("word_pos.csv")

In [99]:
row_count = len(data)
print(f"Number of rows: {row_count}")

Number of rows: 7799


In [100]:
#Clean data
def clean_word(word):
    word = word.lower()
    word = re.sub(r'[^a-z]', '', word)  # keep only letters
    return word

data['clean_word'] = data['headword'].apply(clean_word)
dd['word'] = dd['word'].str.lower()

In [101]:
data.head(10)

,headword,pos,CEFR,CoreInventory 1,CoreInventory 2,Threshold,clean_word
0,a,determiner,A1,NaN,NaN,NaN,a
1,a.m./A.M./am/AM,adverb,A1,NaN,NaN,NaN,amamamam
2,abandon,verb,B1,NaN,NaN,NaN,abandon
3,abandoned,adjective,B2,NaN,NaN,NaN,abandoned
4,ability,noun,A2,NaN,NaN,NaN,ability
5,able,adjective,B1,NaN,NaN,NaN,able
6,abnormal,adjective,B1,NaN,NaN,NaN,abnormal
7,abnormally,adverb,B2,NaN,NaN,NaN,abnormally
8,aboard,adverb,B1,NaN,NaN,NaN,aboard
9,abolish,verb,B2,NaN,NaN,NaN,abolish


In [120]:
data['CEFR'].value_counts()

CEFR
B2    2778
B1    2446
A2    1411
A1    1164
Name: count, dtype: int64

In [102]:
#Merge the two datasets 
data = data.merge(dd, left_on='clean_word', right_on='word', how='left')

In [103]:
data.head()

,headword,pos,CEFR,CoreInventory 1,CoreInventory 2,Threshold,clean_word,word,count
0,a,determiner,A1,NaN,NaN,NaN,a,a,9.081175e+09
1,a.m./A.M./am/AM,adverb,A1,NaN,NaN,NaN,amamamam,NaN,NaN
2,abandon,verb,B1,NaN,NaN,NaN,abandon,abandon,2.747961e+06
3,abandoned,adjective,B2,NaN,NaN,NaN,abandoned,abandoned,6.371760e+06
4,ability,noun,A2,NaN,NaN,NaN,ability,ability,5.200429e+07


In [104]:
#Handle missing values 
data['count'] = data['count'].fillna(1)

In [105]:
#Convert to usable feature 
#Raw counts are too large - use log:

data['log_frequency'] = np.log1p(data['count'])

In [106]:
#dd1.head()

In [107]:
#dd2.head()

In [108]:
#Merge the two datasets on wordID "words.csv" and "word_pos.csv."
#merged_data = pd.merge(
   #    dd1[['word_id', 'word']], 
     #   dd2[['word_id', 'frequency_count', 'level']], 
   #     on='word_id'
  #  )
# 3. Sort by frequency so the most common words are at the top
# Make sure this line has exactly 0 or 4 spaces depending on your block
#merged_data = merged_data.sort_values(by='frequency_count', ascending=False)

# 4. Display the results
#print("Total words merged:", len(merged_data))
#print(merged_data.head())

In [109]:
#row_count = len(merged_data)
#print(f"Number of rows: {row_count}")

In [110]:
#Feature Engineering
#Word Length
data['word_length'] = data['headword'].apply(len)

#Count Vowels
def count_vowels(headword):
    vowels = 'aeiou'
    return sum(1 for letter in headword.lower() if letter in vowels)

data['vowel_count'] = data['headword'].apply(count_vowels)


#Count Syllables
def count_syllables(headword):
  headword = headword.lower()
  syllables = re.findall(r'[aeiou]+', headword)
  return len(syllables)

data['syllable_count'] = data['headword'].apply(count_syllables)

#One-hot encode POS
data = pd.get_dummies(data, columns=['pos'])

In [111]:
#Train/Test Split

#x = data[['word_length','vowel_count','syllable_count']]
#X = data.drop(columns=['heasdword', 'CEFR', 'CoreInventory 1', 'CoreInventory 2', 'Threshold'])
#y = data['CEFR']
x = data[['word_length','vowel_count','syllable_count','log_frequency']]

# Add POS columns automatically
pos_cols = [col for col in data.columns if col.startswith('pos_')]
x = pd.concat([x, data[pos_cols]], axis=1)

y = data['CEFR']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [112]:
#Train Model

model = RandomForestClassifier()
model.fit(x_train, y_train)

joblib.dump(model, "esl_vocab_model.pkl")

['esl_vocab_model.pkl']

In [113]:
#Evaluate Model

predictions = model.predict(x_test)
print(classification_report(y_test, predictions, zero_division=0))

              precision    recall  f1-score   support

          A1       0.41      0.40      0.41       225
          A2       0.23      0.23      0.23       279
          B1       0.35      0.33      0.34       492
          B2       0.48      0.50      0.49       564

    accuracy                           0.38      1560
   macro avg       0.37      0.37      0.37      1560
weighted avg       0.38      0.38      0.38      1560



In [114]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [115]:
X_train,X_test,y_train,y_test = train_test_split(
    x,y,test_size = 0.2, random_state =42)

In [116]:
model = DecisionTreeClassifier()
model.fit(X_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [117]:
y_pred = model.predict(X_test)

In [118]:
accuracy = accuracy_score(y_test,y_pred)
print(accuracy)

0.3871794871794872


In [119]:
#Example Prediction

#new_word ="meticulous"

#length = len(new_word)
#vowels = count_vowels(new_word)
#syllables = count_syllables(new_word)

#new_data = pd.DataFrame({
   # 'word_length' : [length],
  #  'vowel_count' : [vowels],
 #   'syllable_count' : [syllables]
#})

#prediction = model.predict(new_data)
#print("predicted level:",prediction[0])